# What Does a REAL Tree Cost? (Phase C — cost dry run)

Our sweep (`granularity_sweep.run_sweep`) scores leaf sizes over a **flat FAISS index** — `build_flat_retriever`, no RAPTOR tree is ever built. But the paper claims to study *hierarchical* leaf granularity ("RAPTOR is our frozen backbone; we vary only its leaf chunker"). The tree is the thing that makes leaf size a hierarchical question, and it was never in the loop.

The harness section justifies the flat proxy as *"prohibitive on a free-tier budget"* — a claim we never measured. **This notebook measures it.**

The accounting that matters: the coverage proxy (`evidence_coverage`) is pure token overlap and needs **no LLM**, so going hierarchical costs *tree construction only* — zero QA calls. And `construct_tree` summarizes once per **cluster** per layer, stopping once a layer is down to `reduction_dimension + 1` = **11 nodes** — far cheaper than "one call per leaf" intuition suggests.

**Offline by default**: an extractive stand-in replaces the summarizer, so the call count and tree shape are real without an API key or a cent spent. Set `LIVE = True` to bill the real model.

| what we get | why it matters |
|---|---|
| **calls/doc** | the real cost — replaces our estimate |
| **leaves/doc, layers** | the tree-depth confound (see the last cell) |
| **projection** | GO/NO-GO for rebuilding the paper hierarchically |

### Runs on Colab **or** Kaggle

**Kaggle setup — do this first, cell 1 fails without it:**
1. **Settings → Internet → On** (needs a phone-verified account). The clone and the QASPER download both need it.
2. **Settings → Accelerator → GPU T4** — optional; SBERT embeds ~3k leaves here, so CPU works but drags.
3. Only for `LIVE = True`: **Add-ons → Secrets** → add `OPENROUTER_API_KEY`.

Results land in `/kaggle/working/raptor_runs/` (persisted as notebook output). On Colab they go to Drive. Cell 2 picks the right one automatically.


In [ ]:
# 1) Code (ckraptor branch — must include experiments/tree_cost_dryrun.py).
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

# Sanity: the dry-run module must be present, else you cloned an old cache.
from experiments.tree_cost_dryrun import (
    CountingSummarizer, OfflineSummarizer, probe_document, project,
)
print('tree_cost_dryrun present \u2705')

In [ ]:
# 2) Config + persisted output dir (Colab Drive / Kaggle working / local).
N_DOCS  = 5                              # probe size; 5 averages out QASPER length variance
SIZES   = [50, 100, 150, 200, 300, 400]  # the paper's sweep grid
TARGETS = [50, 416]                      # project to: H1/H2 cohort, full QASPER test split
LIVE    = False                          # True -> bill real gpt-oss-120b (needs OPENROUTER_API_KEY)

import os

def run_dir():
    """Where results survive the session: Drive on Colab, /kaggle/working on Kaggle."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return '/content/drive/MyDrive/raptor_runs'
    except Exception:
        pass
    if os.path.isdir('/kaggle/working'):      # Kaggle persists /kaggle/working as output
        return '/kaggle/working/raptor_runs'
    return 'raptor_runs'

RUN_DIR = run_dir()
os.makedirs(RUN_DIR, exist_ok=True)
OUT = os.path.join(RUN_DIR, 'tree_cost_dryrun.json')
print('OUT =', OUT)

In [ ]:
# 3) Load QASPER test docs (the same split the paper's decomposition uses).
from experiments.datasets import get_loader
import statistics

docs = get_loader('qasper').load(limit=N_DOCS)
lens = [len(d.text.split()) for d in docs]
print(f'{len(docs)} docs | median {statistics.median(lens):.0f} words, '
      f'range {min(lens)}-{max(lens)}')
print('\nPer-doc word counts:', lens)

In [ ]:
# 4) PROBE: build a REAL RAPTOR tree per (doc, size) and count summarization calls.
#    ~6 tree builds per doc. Offline this is SBERT + clustering only (no network).
from experiments.config import ExperimentConfig
from tqdm.auto import tqdm

cfg = ExperimentConfig()

def api_key():
    """OPENROUTER_API_KEY from Colab secrets, Kaggle secrets, or the environment."""
    try:
        from google.colab import userdata
        return userdata.get('OPENROUTER_API_KEY')
    except Exception:
        pass
    try:                                        # Kaggle: Add-ons -> Secrets
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    except Exception:
        pass
    return os.environ.get('OPENROUTER_API_KEY')

if LIVE:
    key = api_key()
    if not key:
        raise SystemExit('LIVE=True needs OPENROUTER_API_KEY (Colab secrets / Kaggle Secrets / env)')
    os.environ['OPENROUTER_API_KEY'] = key
    from experiments.models import build_models
    inner, qa_model = build_models(cfg)
else:
    inner, qa_model = OfflineSummarizer(), None

summarizer = CountingSummarizer(inner)
rows = []
for i, doc in enumerate(docs, 1):
    tqdm.write(f'[dryrun] doc {i}/{len(docs)} {doc.doc_id}: building {len(SIZES)} trees')
    rows.extend(probe_document(doc, SIZES, summarizer, cfg, qa_model))

mode = 'LIVE (billed)' if LIVE else 'OFFLINE (extractive stand-in)'
print(f'\n{len(rows)} (doc, size) tree builds | {summarizer.calls} summarization calls | {mode}')

In [ ]:
# 5) THE ANSWER. Per-size cost + tree shape, then the projection to full scale.
import json

by_size = {}
for r in rows:
    a = by_size.setdefault(r['size'], {'calls': 0, 'leaves': 0, 'layers': []})
    a['calls'] += r['calls']
    a['leaves'] += r['n_leaves']
    if r['n_layers'] is not None:
        a['layers'].append(r['n_layers'])

n = len(docs)
print(f"{'leaf size':>10}{'calls/doc':>11}{'leaves/doc':>12}{'layers':>9}")
print('-' * 42)
for s in sorted(by_size):
    a = by_size[s]
    lay = f"{sum(a['layers']) / len(a['layers']):.1f}" if a['layers'] else '-'
    print(f"{s:>10}{a['calls'] / n:>11.1f}{a['leaves'] / n:>12.1f}{lay:>9}")

print(f"\ntotal across all {len(SIZES)} sizes: {sum(r['calls'] for r in rows) / n:.1f} calls/doc\n")
print(f"{'target':>8}{'calls':>10}{'paid USD':>11}{'free-tier days':>16}")
print('-' * 45)
for t in TARGETS:
    p = project(rows, n, t)
    print(f"{p['n_docs']:>8}{p['calls']:>10}{p['usd_paid']:>11.2f}{p['free_tier_days']:>16}")

payload = {'rows': rows, 'n_probed': n, 'live': LIVE,
           'projections': [project(rows, n, t) for t in TARGETS]}
with open(OUT, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'\nwrote {OUT}')

## How to read this

**The cost column is the GO/NO-GO.** If the 416-doc projection is a few dollars, then the paper's *"prohibitive on a free-tier budget"* justification for the flat proxy is wrong, and the flat/hierarchical mismatch — currently the paper's most reject-worthy flaw — is fixable for pocket change. Free-tier is 20 req/min and 1000 req/day (after a one-time \$10 credit purchase); the paid run is likely cheaper than the credit that unlocks the free tier.

**The `layers` column is the science.** `construct_tree` stops once a layer is down to 11 nodes, so on a ~5k-token paper:

| leaf size | ~leaves | tree |
|---|---|---|
| 50 | ~100 | multi-layer |
| 200 | ~25 | shallow |
| 400 | ~13 | ~1 layer / degenerate |

If that pattern shows up in the measured `layers` column, then **leaf size and tree depth are confounded** — at coarse sizes RAPTOR nearly collapses to flat retrieval, at fine sizes it builds a deep summarized hierarchy. The flat proxy cannot see this: it treats a 400-token chunk as merely a coarser retrieval unit.

That has a sharp implication for H1. Our "best fixed size = 200 tokens" may be a **tree-depth optimum wearing a chunk-size costume** — and the held-out `document = -0.045` component could look entirely different once the tree is real.

**Next**: if the cost clears, re-run `granularity_sweep` through `build_answerer("token", ...)` instead of `build_flat_retriever`, scoring with the same `evidence_coverage` proxy. n=50 first as the GO/NO-GO on whether the negative result survives a real tree; then 416 for the decomposition.
